In [29]:
from openai import OpenAI
from pathlib import Path
import sys
import json
import re
import os
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import helpers.scrapers as hs

In [30]:
import importlib
importlib.reload(hs)

<module 'helpers.scrapers' from '/Users/milenakowalska/Desktop/AI/portfolio/ai_summarizer/helpers/scrapers.py'>

In [31]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="local")
MODEL = "llama3"

In [37]:
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')
groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
MODEL_GROQ="openai/gpt-oss-20b"

In [34]:
content_and_links = hs.fetch_website_content_and_links("https://www.nationalgeographic.com/travel/article/stanley-tucci-italy-five-food-regions")

In [12]:
content_and_links

{'title': 'Stanley Tucci explores Italy through its most timeless dishes | National Geographic',
 'content': "TRAVEL\nStanley Tucci explores Italy through its most timeless dishes\nThe second season of\nTucci in Italy\nfollows the actor and best-selling memoirist through Italy’s markets, kitchens, and family recipes in search of authentic flavor.\nStanley Tucci stands in a blood orange orchard in Sicily for the National Geographic series\nTucci in Italy\n.\nNational Geographic/Matt Holyoak\nBy\nElena Giardina\nLast updated May 12, 2026\nStanley Tucci\nis no stranger to the wonders and complexities of Italy.\nHe’s\ntraversed the country’s\nmost famous and obscure locales\n(\nFlorence\nand\nRome\n,\nMaremma and Senarica)\nand has\nwritten about its\nirresistible food\nin his cookbooks and memoirs. But\nthere are always new things to\ndiscover\nin\na place where culture and cuisine are inseparable.\nIn a second season of\nTucci in Italy\n, the\nactor returns to his\nancestral homeland to 

In [35]:
link_system_prompt = """
    You are provided with an article title and a list of links found on a webpage.
    You are able to decide which of the links would be most relevant to include in a summary of the given article,
    such as links to an author, or pages clarifying the history or the context of the topic.
    Please note that if you chose a relative link as relevant, you should build a full url out of it. 
    For example, if the chosen url is "/about-author" and the website you are analyzing is https://full.url/article-123, 
    you should build as a response https://full.url/about-author.
    You should respond in JSON as in this example (replace topic_name with the actual topic):

    {
        "links": [
            {"type": "author page", "url": "https://full.url/goes/here/about"},
            {"type": "context - topic (topic_name)", "url": "https://another.full.url/context"}
        ]
    }
"""

In [36]:
def get_links_user_prompt(content_and_links):
    user_prompt = f"""
You are preparing context in order to summarize the article with the title: {content_and_links["title"]}.
Here is the list of links on the website  {content_and_links["url"]} -
Please decide which of these are relevant web links for a summary of the article, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    user_prompt += "\n".join(content_and_links["links"])
    return user_prompt

In [15]:
def select_relevant_links(content_and_links):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(content_and_links)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [38]:
def select_relevant_links_groq(content_and_links):
    response = groq.chat.completions.create(
        model=MODEL_GROQ,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(content_and_links)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [84]:
select_relevant_links(content_and_links)

{'links': [{'type': 'author page',
   'url': 'https://www.nationalgeographic.com/travel/article/stanley-tucci-interview-tucci-in-italy'},
  {'type': 'related topic - food culture',
   'url': 'https://www.nationalgeographic.com/related/d9881753-b9f1-37a5-81e3-90ce36e10a3c/food-culture'}]}

In [44]:
def fetch_page_and_all_relevant_links(content_and_links, model):
    if model == "groq":
        relevant_links = select_relevant_links_groq(content_and_links)
    else:
        relevant_links = select_relevant_links(content_and_links)
    result = f"##Article title: \n\n{content_and_links["title"]} \n##Landing Page:\n\n{content_and_links["content"]}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += hs.fetch_website_content(link["url"])
    return result

In [45]:
print(fetch_page_and_all_relevant_links(content_and_links, "groq"))

##Article title: 

Stanley Tucci explores Italy through its most timeless dishes | National Geographic 
##Landing Page:

TRAVEL
Stanley Tucci explores Italy through its most timeless dishes
The second season of
Tucci in Italy
follows the actor and best-selling memoirist through Italy’s markets, kitchens, and family recipes in search of authentic flavor.
Stanley Tucci stands in a blood orange orchard in Sicily for the National Geographic series
Tucci in Italy
.
National Geographic/Matt Holyoak
By
Elena Giardina
Last updated May 12, 2026
Stanley Tucci
is no stranger to the wonders and complexities of Italy.
He’s
traversed the country’s
most famous and obscure locales
(
Florence
and
Rome
,
Maremma and Senarica)
and has
written about its
irresistible food
in his cookbooks and memoirs. But
there are always new things to
discover
in
a place where culture and cuisine are inseparable.
In a second season of
Tucci in Italy
, the
actor returns to his
ancestral homeland to explore Naples and Campa

In [46]:
tutorial_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages to a given main article/topic
and creates a short summary / learning tutorial about the topic and its context.
Respond in markdown without code blocks.
Include details of the history, article's author and wide context of the topic, but only if you have the information.
"""

In [47]:
def get_tutorial_user_prompt(url, model):
    content_and_links = hs.fetch_website_content_and_links(url)
    page_and_links = fetch_page_and_all_relevant_links(content_and_links, model)
    user_prompt = f"""
You are looking at a company called: {content_and_links["title"]}
Here are the contents of the article's landing page and other relevant pages;
use this information to build a short summary / learning tutorial about the main topics from the article.\n\n
"""
    user_prompt += page_and_links
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [48]:
get_tutorial_user_prompt("https://www.nationalgeographic.com/travel/article/stanley-tucci-italy-five-food-regions", "groq")

"\nYou are looking at a company called: Stanley Tucci explores Italy through its most timeless dishes | National Geographic\nHere are the contents of the article's landing page and other relevant pages;\nuse this information to build a short summary / learning tutorial about the main topics from the article.\n\n\n##Article title: \n\nStanley Tucci explores Italy through its most timeless dishes | National Geographic \n##Landing Page:\n\nTRAVEL\nStanley Tucci explores Italy through its most timeless dishes\nThe second season of\nTucci in Italy\nfollows the actor and best-selling memoirist through Italy’s markets, kitchens, and family recipes in search of authentic flavor.\nStanley Tucci stands in a blood orange orchard in Sicily for the National Geographic series\nTucci in Italy\n.\nNational Geographic/Matt Holyoak\nBy\nElena Giardina\nLast updated May 12, 2026\nStanley Tucci\nis no stranger to the wonders and complexities of Italy.\nHe’s\ntraversed the country’s\nmost famous and obscur

In [49]:
def slugify_filename(title: str, max_length: int = 120) -> str:
    title = title.strip().lower()
    title = re.sub(r"[^\w\s-]", "", title)
    title = re.sub(r"[\s_-]+", "-", title)
    title = title.strip("-")
    return title[:max_length] or "summary"

In [50]:
def create_tutorial(url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": tutorial_system_prompt},
            {"role": "user", "content": get_tutorial_user_prompt(url, "ollama")}
        ],
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [60]:
def stream_tutorial_groq(url):
    stream = groq.chat.completions.create(
        model=MODEL_GROQ,
        messages=[
            {"role": "system", "content": tutorial_system_prompt},
            {"role": "user", "content": get_tutorial_user_prompt(url, "groq")}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [57]:
def create_tutorial_save(url):
    result = create_tutorial(url)
    title_match = re.search(r"^#\s+(.+)$", result, flags=re.MULTILINE)
    title = title_match.group(1) if title_match else "tutorial"

    summaries_dir = PROJECT_ROOT / "summaries"
    summaries_dir.mkdir(parents=True, exist_ok=True)

    output_path = summaries_dir / f"{slugify_filename(title)}.md"
    output_path.write_text(result, encoding="utf-8")

    return output_path

In [ ]:
create_tutorial_save("https://www.nationalgeographic.com/travel/article/stanley-tucci-italy-five-food-regions")

PosixPath('/Users/milenakowalska/Desktop/AI/portfolio/ai_summarizer/summaries/tutorial.md')

In [58]:
# UI
import gradio as gr

In [61]:
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_tutorial_groq,
    title="Tutorial Generator", 
    inputs=[url_input], 
    outputs=[message_output], 
    examples=[
            ["https://www.nationalgeographic.com/travel/article/stanley-tucci-italy-five-food-regions"],
            ["https://www.nationalgeographic.com/history/article/franklin-expedition-sailors-identified"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.
